# Análisis de Caso: Preprocesamiento y Escalamiento de Datos

**Objetivo:** Desarrollar un proceso completo de preprocesamiento de datos para la empresa RetailData Analytics, con el fin de preparar un conjunto de datos demográficos y de comportamiento de clientes para un modelo predictivo. 

**Técnicas a aplicar:**
* Imputación de valores faltantes (Media).
* Codificación de variables categóricas (Label Encoding, One-Hot Encoding, Variables Dummy).
* Escalamiento de variables numéricas (Normalización Min-Max, Estandarización Z-Score).

## Introducción Teórica

El **preprocesamiento de datos** es una etapa fundamental en el desarrollo de proyectos de Machine Learning. Consiste en limpiar, transformar y organizar los datos crudos para que los algoritmos puedan interpretarlos de manera eficiente y precisa.

* **Importancia:** Los modelos de Machine Learning son tan buenos como los datos con los que se entrenan. Datos ruidosos o incompletos generan modelos deficientes.
* **Valores faltantes (Nulos):** Son ausencias de información en ciertas observaciones (ej. un cliente no registró sus ingresos). Deben ser tratados (eliminados o imputados) porque la mayoría de los algoritmos matemáticos no los soportan.
* **Variables categóricas:** Son datos cualitativos (texto o categorías, como la "Ciudad"). Deben transformarse a números para que el modelo pueda procesarlas matemáticamente.
* **Escalamiento de variables numéricas:** Consiste en ajustar el rango de los datos numéricos continuos. Variables con escalas muy diferentes (ej. Edad vs. Ingresos) pueden sesgar el modelo hacia la variable con magnitudes mayores.

In [2]:
# 3. Importación de librerías
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler, StandardScaler
import warnings

# Ignorar advertencias menores para mantener el notebook limpio
warnings.filterwarnings('ignore')

In [3]:
# 4. Creación del dataset
# Basado en la tabla proporcionada en el caso de estudio
data = {
    'ID': [1, 2, 3, 4],
    'Edad': [25, 45, 30, 40],
    'Ciudad': ['Madrid', 'Sevilla', 'Madrid', 'Barcelona'],
    'Ingresos': [30000.0, 50000.0, np.nan, 40000.0]
}

df = pd.DataFrame(data)

# Mostrar el DataFrame original
print("DataFrame Original:")
display(df)

DataFrame Original:


,ID,Edad,Ciudad,Ingresos
0,1,25,Madrid,30000.0
1,2,45,Sevilla,50000.0
2,3,30,Madrid,NaN
3,4,40,Barcelona,40000.0


## Exploración inicial de los datos

Antes de aplicar cualquier transformación, es vital conocer la estructura de nuestro dataset. A continuación, revisaremos:
1. La visualización general del DataFrame.
2. Los tipos de datos de cada columna.
3. La cantidad de valores nulos (faltantes) presentes.
4. Un resumen estadístico simple de las variables numéricas.

In [4]:
# 5. Exploración inicial de los datos
print("Información general del dataset:")
df.info()

print("\nConteo de valores nulos por columna:")
print(df.isnull().sum())

print("\nResumen estadístico del dataset:")
display(df.describe())

Información general del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ID        4 non-null      int64  
 1   Edad      4 non-null      int64  
 2   Ciudad    4 non-null      str    
 3   Ingresos  3 non-null      float64
dtypes: float64(1), int64(2), str(1)
memory usage: 260.0 bytes

Conteo de valores nulos por columna:
ID          0
Edad        0
Ciudad      0
Ingresos    1
dtype: int64

Resumen estadístico del dataset:


,ID,Edad,Ingresos
count,4.000000,4.000000,3.0
mean,2.500000,35.000000,40000.0
std,1.290994,9.128709,10000.0
min,1.000000,25.000000,30000.0
25%,1.750000,28.750000,35000.0
50%,2.500000,35.000000,40000.0
75%,3.250000,41.250000,45000.0
max,4.000000,45.000000,50000.0


## Imputación de valores faltantes

Los valores nulos pueden generar errores durante el entrenamiento de un modelo de Machine Learning. En este caso, la columna `Ingresos` presenta un valor faltante (`NaN`). 

Para solucionarlo, utilizaremos la **imputación por la media**. Esto significa que calcularemos el promedio de los ingresos existentes y usaremos ese valor para rellenar el dato faltante. Es una técnica útil cuando la variable tiene una distribución relativamente normal y no posee valores atípicos (outliers) extremos.

In [5]:
# 6. Imputación de valores faltantes en la columna Ingresos utilizando la media
# Inicializamos el imputador
imputer = SimpleImputer(strategy='mean')

# Aplicamos la imputación manteniendo la estructura de DataFrame
df['Ingresos_Imputados'] = imputer.fit_transform(df[['Ingresos']])

print("DataFrame tras la imputación (se conserva la columna original para comparación):")
display(df[['ID', 'Ingresos', 'Ingresos_Imputados']])

DataFrame tras la imputación (se conserva la columna original para comparación):


,ID,Ingresos,Ingresos_Imputados
0,1,30000.0,30000.0
1,2,50000.0,50000.0
2,3,NaN,40000.0
3,4,40000.0,40000.0


## Codificación de la variable categórica: Label Encoding

Los algoritmos matemáticos requieren entradas numéricas. La columna `Ciudad` contiene texto. 

El **Label Encoding** asigna un número entero único a cada categoría distinta (ej. Barcelona=0, Madrid=1, Sevilla=2). 
* *Nota importante:* Esta técnica introduce un orden numérico artificial. Aunque puede ser útil para variables ordinales (como "Bajo", "Medio", "Alto"), en variables nominales (como ciudades) el modelo podría interpretar erróneamente que "Sevilla" (2) vale más que "Barcelona" (0).

In [6]:
# 7. Transformar la columna Ciudad utilizando Label Encoding
le = LabelEncoder()

# Creamos una nueva columna con los valores codificados
df['Ciudad_Label'] = le.fit_transform(df['Ciudad'])

print("DataFrame tras aplicar Label Encoding:")
display(df[['ID', 'Ciudad', 'Ciudad_Label']])

# Mostrar el mapeo exacto generado por el codificador
print("\nMapeo de Label Encoding:")
for idx, clase in enumerate(le.classes_):
    print(f"{clase} -> {idx}")

DataFrame tras aplicar Label Encoding:


,ID,Ciudad,Ciudad_Label
0,1,Madrid,1
1,2,Sevilla,2
2,3,Madrid,1
3,4,Barcelona,0



Mapeo de Label Encoding:
Barcelona -> 0
Madrid -> 1
Sevilla -> 2


## Codificación de la variable categórica: One-Hot Encoding

Para evitar el problema de jerarquía artificial del Label Encoding en variables nominales, utilizamos **One-Hot Encoding**. 

Esta técnica crea una nueva columna binaria (0 o 1) para cada categoría única presente en la variable original. Si una fila pertenece a "Madrid", tendrá un 1 en la columna de Madrid y 0 en el resto.

In [7]:
# 8. Transformar la columna Ciudad utilizando One-Hot Encoding de Scikit-Learn
ohe = OneHotEncoder(sparse_output=False, dtype=int)

# Entrenar y transformar la columna
ohe_array = ohe.fit_transform(df[['Ciudad']])

# Obtener los nombres de las nuevas columnas
ohe_columns = ohe.get_feature_names_out(['Ciudad'])

# Crear un DataFrame temporal con los resultados
df_ohe = pd.DataFrame(ohe_array, columns=ohe_columns)

print("Resultado de One-Hot Encoding:")
display(df_ohe)

# Unimos estas nuevas columnas al DataFrame principal
df = pd.concat([df, df_ohe], axis=1)

Resultado de One-Hot Encoding:


,Ciudad_Barcelona,Ciudad_Madrid,Ciudad_Sevilla
0,0,1,0
1,0,0,1
2,0,1,0
3,1,0,0


## Aplicación de Variables Dummy

El método de **Variables Dummy** es conceptualmente idéntico al One-Hot Encoding, pero generalmente se realiza usando Pandas (`pd.get_dummies()`). 

La diferencia clave en la práctica habitual es que al crear variables Dummy, solemos **eliminar la primera categoría** (`drop_first=True`) para evitar la "trampa de las variables ficticias" (multicolinealidad). Si tenemos 3 ciudades, solo necesitamos 2 columnas para representarlas (si ambas son 0, implícitamente es la tercera ciudad).

In [8]:
# 9. Aplicar el método de Variables Dummy para la columna Ciudad
# Usamos drop_first=True para evitar la multicolinealidad
df_dummies = pd.get_dummies(df['Ciudad'], prefix='Dummy_Ciudad', drop_first=True, dtype=int)

print("Resultado de pd.get_dummies con drop_first=True:")
display(df_dummies)

# Unimos al DataFrame principal
df = pd.concat([df, df_dummies], axis=1)

Resultado de pd.get_dummies con drop_first=True:


,Dummy_Ciudad_Madrid,Dummy_Ciudad_Sevilla
0,1,0
1,0,1
2,1,0
3,0,0


## Escalamiento de variables numéricas

Las columnas `Edad` e `Ingresos` manejan rangos de valores completamente distintos (decenas vs. decenas de miles). Si no se escalan, un modelo basado en distancias le dará demasiada importancia a los ingresos simplemente por ser números más grandes.

### A. Normalización Min-Max
Transforma los datos para que todos caigan en un rango específico, comúnmente entre 0 y 1. Es útil cuando no conocemos la distribución de los datos.

Fórmula:
$$X_{norm}=\frac{X-X_{min}}{X_{max}-X_{min}}$$

### B. Estandarización Z-Score
Transforma los datos para que tengan una media ($\mu$) de 0 y una desviación estándar ($\sigma$) de 1. Es ideal cuando los datos siguen una distribución normal o cuando hay presencia de valores atípicos (ya que la normalización Min-Max es muy sensible a ellos).

Fórmula:
$$Z=\frac{X-\mu}{\sigma}$$

In [9]:
# 10. Escalar las columnas Edad e Ingresos

# Variables a escalar (usamos los ingresos imputados)
cols_to_scale = ['Edad', 'Ingresos_Imputados']

# A. Normalización Min-Max
minmax_scaler = MinMaxScaler()
df[['Edad_MinMax', 'Ingresos_MinMax']] = minmax_scaler.fit_transform(df[cols_to_scale])

# B. Estandarización Z-Score
std_scaler = StandardScaler()
df[['Edad_ZScore', 'Ingresos_ZScore']] = std_scaler.fit_transform(df[cols_to_scale])

print("Vista previa de las variables escaladas:")
display(df[['Edad', 'Edad_MinMax', 'Edad_ZScore', 'Ingresos_Imputados', 'Ingresos_MinMax', 'Ingresos_ZScore']])

Vista previa de las variables escaladas:


,Edad,Edad_MinMax,Edad_ZScore,Ingresos_Imputados,Ingresos_MinMax,Ingresos_ZScore
0,25,0.00,-1.264911,30000.0,0.0,-1.414214
1,45,1.00,1.264911,50000.0,1.0,1.414214
2,30,0.25,-0.632456,40000.0,0.5,0.000000
3,40,0.75,0.632456,40000.0,0.5,0.000000


## Comparación de resultados

A continuación, visualizamos una tabla consolidada comparando los valores originales (o imputados) frente a sus transformaciones. 

* **Min-Max:** Podemos observar que el valor mínimo siempre se convierte en `0.0` y el máximo en `1.0`.
* **Z-Score:** Los valores resultantes pueden ser negativos (si están por debajo de la media) o positivos (si están por encima), y la magnitud representa cuántas desviaciones estándar se alejan de la media.

In [10]:
# 11. Tabla final de comparación de variables numéricas
comparacion_df = df[[
    'Edad', 'Edad_MinMax', 'Edad_ZScore', 
    'Ingresos_Imputados', 'Ingresos_MinMax', 'Ingresos_ZScore'
]].copy()

# Renombramos para mayor claridad en la presentación
comparacion_df.rename(columns={'Ingresos_Imputados': 'Ingresos'}, inplace=True)

display(comparacion_df)

,Edad,Edad_MinMax,Edad_ZScore,Ingresos,Ingresos_MinMax,Ingresos_ZScore
0,25,0.00,-1.264911,30000.0,0.0,-1.414214
1,45,1.00,1.264911,50000.0,1.0,1.414214
2,30,0.25,-0.632456,40000.0,0.5,0.000000
3,40,0.75,0.632456,40000.0,0.5,0.000000


## Dataset final transformado

Finalmente, prepararemos la versión limpia de nuestro conjunto de datos. Mantendremos únicamente las columnas transformadas que serían de utilidad para un modelo de Machine Learning y exportaremos el resultado.

In [11]:
# 12. Dataset final transformado
# Seleccionamos las columnas definitivas. Asumamos que para el modelo usaremos
# la edad y los ingresos estandarizados (Z-Score), y las variables Dummy para la ciudad.

columnas_finales = [
    'ID', 
    'Edad_ZScore', 
    'Ingresos_ZScore', 
    'Dummy_Ciudad_Madrid', 
    'Dummy_Ciudad_Sevilla'
]

df_final = df[columnas_finales]

print("Dataset Final Preparado para Modelado:")
display(df_final)

# Exportar a CSV y Excel
df_final.to_csv('datos_preprocesados.csv', index=False)
df_final.to_excel('datos_preprocesados.xlsx', index=False)
print("\nArchivos 'datos_preprocesados.csv' y 'datos_preprocesados.xlsx' guardados con éxito.")

Dataset Final Preparado para Modelado:


,ID,Edad_ZScore,Ingresos_ZScore,Dummy_Ciudad_Madrid,Dummy_Ciudad_Sevilla
0,1,-1.264911,-1.414214,1,0
1,2,1.264911,1.414214,0,1
2,3,-0.632456,0.000000,1,0
3,4,0.632456,0.000000,0,0



Archivos 'datos_preprocesados.csv' y 'datos_preprocesados.xlsx' guardados con éxito.
